In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

from armored.models import *
from armored.preprocessing import *

from sklearn.model_selection import KFold

import itertools

from tqdm import tqdm

/home/jcthompson5@ad.wisc.edu/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# data with initial and end point measurements
df_exp0 = pd.read_csv("data/exp0/exp0_metabolites.csv")
df_exp1 = pd.read_csv("data/exp1/exp1_metabolites.csv")
df_exp2 = pd.read_csv("data/exp2/exp2_metabolites.csv")
df_exp3 = pd.read_csv("data/exp3/exp3_metabolites.csv")

In [3]:
species = ['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs', 'CHabs',
           'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs', 'RIabs']
controls = ['AcGum', 'ArGal', 'Inulin', 'Pectin', 'Starch', 'Xylan']
# only measured pH in dtl 0
metabolites = ['pH']

# concatenate all observed and all system variables 
observed = np.concatenate((np.array(species), np.array(metabolites)))
system_variables = np.concatenate((np.array(species), np.array(metabolites), np.array(controls)))
system_variables

array(['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs',
       'CHabs', 'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs',
       'RIabs', 'pH', 'AcGum', 'ArGal', 'Inulin', 'Pectin', 'Starch',
       'Xylan'], dtype='<U6')

In [4]:
# separate monoculture data (monoculture always trained on, never used as held-out data)
df_mono = []
df_comm = []

for exp, df_exp in df_exp0.groupby("Experiments"):
    
    # check if monoculture or not
    if sum(df_exp.iloc[df_exp.Time.values==0][species].values[0] > 0) > 1:
        df_comm.append(df_exp)
    else:
        df_mono.append(df_exp)
        
# concatenate dfs
df_mono = pd.concat(df_mono)
df_exp0 = pd.concat(df_comm)

# Train on DTL 0 to predict held-out DTL 0, 1, 2, 3

In [5]:
# df for training 
df = df_exp0.copy()
all_treatments = df.Experiments.values
unique_treatments = np.unique(all_treatments) 

n_splits = 20
kf = KFold(n_splits=n_splits, shuffle=True, random_state=21)
k_fold_df = pd.DataFrame()
for i, (train_index, test_index) in enumerate(kf.split(unique_treatments)):

    # split into train and test
    train_index = np.in1d(all_treatments, unique_treatments[train_index])
    test_index  = np.in1d(all_treatments, unique_treatments[test_index])
    train_df = df.iloc[train_index].copy()
    test_df  = df.iloc[test_index].copy() 
    
    # add monoculture data
    train_df = pd.concat((df_mono, train_df))
    
    # scale data 
    scaler = MinMaxScaler(observed, system_variables)
    scaler.fit(train_df)
    train_df_scaled = scaler.transform(train_df.copy())
    test_df_scaled = scaler.transform(test_df.copy())
    
    # format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
    train_data = format_data(train_df, species, metabolites, controls, observed=observed)
    train_data_scaled = format_data(train_df_scaled, species, metabolites, controls, observed=observed)
    test_data = format_data(test_df, species, metabolites, controls, observed=observed)
    test_data_scaled = format_data(test_df_scaled, species, metabolites, controls, observed=observed)

    # instantiate model
    brnn = miRNN(n_species=len(species), 
                 n_metabolites=len(metabolites), 
                 n_controls=len(controls), 
                 n_hidden=32)
    
    # fit model
    brnn.fit(train_data_scaled, 
             alpha_0=0., alpha_1=1.,
             evd_tol=1e-3)

    # make predictions
    predictions = scaler.inverse_transform(brnn.predict(test_data_scaled))

    # save predictions
    pred_df = pd.DataFrame()
    for (T, X, U, Y, exp_names), (_, preds, stdvs, exp_names_pred) in zip(test_data, predictions):

        # save species predictions for each experimental condition
        for i, exp_name in enumerate(exp_names):
            # init dataframe
            pred_df_exp = pd.DataFrame()

            # insert exp name
            pred_df_exp["Experiments"] = [exp_name]*len(T[i])
            pred_df_exp["Time"] = T[i]

            for j, s in enumerate(observed):
                pred_df_exp[s + " true"] = Y[i,:,j]
                pred_df_exp[s + " pred"] = preds[i,:,j]
                pred_df_exp[s + " stdv"] = stdvs[i,:,j]

            # append to test prediction dataframe
            pred_df = pd.concat((pred_df, pred_df_exp))
    k_fold_df = pd.concat((k_fold_df, pred_df))
    k_fold_df.to_csv(f"kfold/MiRNN_20_fold_32h_dtl_0.csv", index=False)
    
# train on dtl 0 to predict dtl 1 and 2
train_df = pd.concat((df_mono, df_exp0))
test_df = pd.concat((df_exp1, df_exp2, df_exp3))

# scale data 
scaler = MinMaxScaler(observed, system_variables)
scaler.fit(train_df)
train_df_scaled = scaler.transform(train_df.copy())
test_df_scaled = scaler.transform(test_df.copy())

# format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
train_data = format_data(train_df, species, metabolites, controls, observed=observed)
train_data_scaled = format_data(train_df_scaled, species, metabolites, controls, observed=observed)
test_data = format_data(test_df, species, metabolites, controls, observed=observed)
test_data_scaled = format_data(test_df_scaled, species, metabolites, controls, observed=observed)

# instantiate model
brnn = miRNN(n_species=len(species), 
             n_metabolites=len(metabolites), 
             n_controls=len(controls), 
             n_hidden=32)

# fit model
brnn.fit(train_data_scaled, 
         alpha_0=0., alpha_1=1.,
         evd_tol=1e-3)

# make predictions
predictions = scaler.inverse_transform(brnn.predict(test_data_scaled))

# save predictions
pred_df = pd.DataFrame()
for (T, X, U, Y, exp_names), (_, preds, stdvs, exp_names_pred) in zip(test_data, predictions):

    # save species predictions for each experimental condition
    for i, exp_name in enumerate(exp_names):
        # init dataframe
        pred_df_exp = pd.DataFrame()

        # insert exp name
        pred_df_exp["Experiments"] = [exp_name]*len(T[i])
        pred_df_exp["Time"] = T[i]

        for j, s in enumerate(observed):
            pred_df_exp[s + " true"] = Y[i,:,j]
            pred_df_exp[s + " pred"] = preds[i,:,j]
            pred_df_exp[s + " stdv"] = stdvs[i,:,j]

        # append to test prediction dataframe
        pred_df = pd.concat((pred_df, pred_df_exp))
k_fold_df = pd.concat((k_fold_df, pred_df))
k_fold_df.to_csv(f"kfold/MiRNN_20_fold_32h_dtl_0.csv", index=False)

Total measurements: 3015, Number of parameters: 2320, Initial regularization: 0.00e+00
Loss: 52.420, Residuals: -0.03676
Loss: 44.998, Residuals: -0.00404
Loss: 36.254, Residuals: -0.00202
Loss: 34.271, Residuals: -0.00167
Loss: 33.787, Residuals: -0.00195
Loss: 32.843, Residuals: -0.00128
Loss: 31.428, Residuals: 0.00406
Loss: 31.118, Residuals: -0.00241
Loss: 30.589, Residuals: -0.00147
Loss: 29.683, Residuals: 0.00015
Loss: 27.975, Residuals: 0.00274
Loss: 27.435, Residuals: 0.00589
Loss: 27.087, Residuals: -0.00076
Loss: 26.540, Residuals: -0.00010
Loss: 22.810, Residuals: 0.00585
Loss: 22.240, Residuals: 0.00120
Loss: 21.728, Residuals: 0.00092
Loss: 21.243, Residuals: 0.00255
Loss: 21.063, Residuals: 0.00412
Loss: 20.895, Residuals: 0.00046
Loss: 20.645, Residuals: 0.00089
Loss: 20.265, Residuals: 0.00099
Loss: 19.754, Residuals: 0.00129
Loss: 19.587, Residuals: 0.00107
Loss: 19.551, Residuals: -0.00008
Loss: 19.210, Residuals: 0.00006
Loss: 18.544, Residuals: 0.00000
Loss: 18.07

Loss: 517.804, Residuals: -0.00012
Loss: 517.770, Residuals: -0.00012
Loss: 517.712, Residuals: -0.00012
Loss: 517.673, Residuals: -0.00012
Loss: 517.624, Residuals: -0.00012
Loss: 517.585, Residuals: -0.00012
Loss: 517.542, Residuals: -0.00012
Loss: 517.512, Residuals: -0.00012
Loss: 517.476, Residuals: -0.00012
Loss: 517.456, Residuals: -0.00012
Loss: 517.426, Residuals: -0.00012
Updating precision...
Evidence 4100.559
Loss: 891.258, Residuals: -0.00012
Loss: 889.462, Residuals: -0.00012
Loss: 886.474, Residuals: -0.00012
Loss: 883.348, Residuals: -0.00012
Loss: 881.584, Residuals: -0.00011
Loss: 879.503, Residuals: -0.00010
Loss: 878.413, Residuals: -0.00011
Loss: 876.687, Residuals: -0.00010
Loss: 875.931, Residuals: -0.00010
Loss: 874.819, Residuals: -0.00010
Loss: 873.949, Residuals: -0.00009
Loss: 872.833, Residuals: -0.00009
Loss: 872.560, Residuals: -0.00009
Loss: 872.070, Residuals: -0.00009
Loss: 871.280, Residuals: -0.00009
Loss: 871.054, Residuals: -0.00009
Loss: 870.643, 

Loss: 1462.430, Residuals: -0.00003
Loss: 1462.284, Residuals: -0.00003
Updating precision...
Evidence 5193.459
Pass count  1
Fail count  2
Total measurements: 3018, Number of parameters: 2320, Initial regularization: 0.00e+00
Loss: 52.438, Residuals: -0.03688
Loss: 44.883, Residuals: -0.00388
Loss: 36.091, Residuals: -0.00189
Loss: 34.167, Residuals: -0.00164
Loss: 33.659, Residuals: -0.00189
Loss: 32.704, Residuals: -0.00132
Loss: 31.162, Residuals: 0.00267
Loss: 30.317, Residuals: 0.01152
Loss: 29.412, Residuals: 0.00074
Loss: 28.939, Residuals: 0.00797
Loss: 28.376, Residuals: 0.00091
Loss: 27.786, Residuals: 0.00279
Loss: 26.852, Residuals: 0.00323
Loss: 26.703, Residuals: -0.00108
Loss: 22.538, Residuals: 0.00337
Loss: 21.637, Residuals: 0.00301
Loss: 21.207, Residuals: 0.00201
Loss: 21.038, Residuals: 0.00049
Loss: 20.724, Residuals: 0.00048
Loss: 20.220, Residuals: 0.00089
Loss: 19.423, Residuals: 0.00150
Loss: 19.342, Residuals: 0.00282
Loss: 19.254, Residuals: 0.00046
Loss: 1

Loss: 532.819, Residuals: -0.00023
Loss: 531.791, Residuals: -0.00023
Loss: 531.247, Residuals: -0.00023
Loss: 530.551, Residuals: -0.00022
Loss: 529.952, Residuals: -0.00022
Loss: 529.412, Residuals: -0.00021
Loss: 529.066, Residuals: -0.00021
Loss: 528.661, Residuals: -0.00021
Loss: 528.383, Residuals: -0.00021
Loss: 527.994, Residuals: -0.00021
Loss: 527.815, Residuals: -0.00021
Loss: 527.538, Residuals: -0.00021
Loss: 527.330, Residuals: -0.00021
Loss: 527.156, Residuals: -0.00020
Loss: 526.970, Residuals: -0.00020
Loss: 526.909, Residuals: -0.00020
Loss: 526.814, Residuals: -0.00020
Loss: 526.705, Residuals: -0.00020
Loss: 526.613, Residuals: -0.00020
Loss: 526.533, Residuals: -0.00020
Loss: 526.477, Residuals: -0.00020
Loss: 526.398, Residuals: -0.00020
Loss: 526.354, Residuals: -0.00020
Loss: 526.306, Residuals: -0.00020
Loss: 526.255, Residuals: -0.00020
Loss: 526.219, Residuals: -0.00020
Loss: 526.182, Residuals: -0.00020
Loss: 526.166, Residuals: -0.00020
Loss: 526.140, Resid

Loss: 26.916, Residuals: 0.00353
Loss: 25.851, Residuals: 0.00625
Loss: 25.267, Residuals: -0.00091
Loss: 24.507, Residuals: 0.00035
Loss: 23.334, Residuals: 0.00123
Loss: 21.997, Residuals: 0.00315
Loss: 21.924, Residuals: 0.00796
Loss: 21.787, Residuals: 0.00707
Loss: 21.555, Residuals: 0.00516
Loss: 21.326, Residuals: 0.00051
Loss: 21.009, Residuals: 0.00076
Loss: 20.460, Residuals: 0.00051
Loss: 19.698, Residuals: 0.00071
Loss: 19.409, Residuals: 0.00222
Loss: 19.361, Residuals: 0.00010
Loss: 18.946, Residuals: 0.00080
Loss: 18.217, Residuals: 0.00092
Loss: 17.893, Residuals: 0.00398
Loss: 17.608, Residuals: -0.00040
Loss: 17.231, Residuals: 0.00026
Loss: 16.600, Residuals: 0.00030
Loss: 16.031, Residuals: 0.00175
Loss: 15.807, Residuals: 0.00068
Loss: 15.473, Residuals: 0.00092
Loss: 15.369, Residuals: 0.00153
Loss: 15.215, Residuals: 0.00089
Loss: 14.966, Residuals: 0.00085
Loss: 14.635, Residuals: 0.00076
Loss: 14.404, Residuals: 0.00105
Loss: 14.103, Residuals: 0.00105
Loss: 14

...

Loss: 12.297, Residuals: 0.00015
Loss: 12.210, Residuals: 0.00015
Loss: 12.055, Residuals: 0.00013
Loss: 11.814, Residuals: -0.00007
Loss: 11.493, Residuals: -0.00009
Loss: 11.129, Residuals: -0.00067
Loss: 11.104, Residuals: -0.00004
Loss: 10.878, Residuals: -0.00002
Loss: 10.529, Residuals: -0.00020
Loss: 10.319, Residuals: -0.00008
Loss: 10.017, Residuals: -0.00029
Loss: 9.714, Residuals: -0.00086
Loss: 9.690, Residuals: 0.00010
Loss: 9.526, Residuals: 0.00007
Loss: 9.304, Residuals: -0.00005
Loss: 9.252, Residuals: -0.00041
Loss: 9.175, Residuals: -0.00036
Loss: 9.155, Residuals: 0.00009
Loss: 9.122, Residuals: 0.00009
Loss: 9.066, Residuals: 0.00008
Loss: 8.979, Residuals: 0.00006
Loss: 8.867, Residuals: 0.00003
Loss: 8.745, Residuals: 0.00005
Loss: 8.714, Residuals: 0.00003
Loss: 8.662, Residuals: 0.00003
Loss: 8.603, Residuals: 0.00001
Loss: 8.583, Residuals: -0.00000
Loss: 8.548, Residuals: -0.00001
Loss: 8.505, Residuals: -0.00002
Loss: 8.475, Residuals: -0.00003
Loss: 8.444, 

Loss: 517.623, Residuals: -0.00010
Loss: 517.582, Residuals: -0.00010
Loss: 517.549, Residuals: -0.00010
Loss: 517.513, Residuals: -0.00010
Loss: 517.488, Residuals: -0.00010
Loss: 517.458, Residuals: -0.00010
Loss: 517.435, Residuals: -0.00010
Loss: 517.408, Residuals: -0.00010
Loss: 517.394, Residuals: -0.00009
Loss: 517.374, Residuals: -0.00009
Updating precision...
Evidence 4144.804
Loss: 894.470, Residuals: -0.00011
Loss: 886.622, Residuals: -0.00009
Loss: 881.081, Residuals: -0.00009
Loss: 876.961, Residuals: -0.00010
Loss: 873.786, Residuals: -0.00008
Loss: 871.690, Residuals: -0.00009
Loss: 869.450, Residuals: -0.00008
Loss: 868.143, Residuals: -0.00006
Loss: 866.492, Residuals: -0.00005
Loss: 866.153, Residuals: -0.00006
Loss: 865.575, Residuals: -0.00006
Loss: 864.776, Residuals: -0.00005
Loss: 864.191, Residuals: -0.00005
Loss: 863.562, Residuals: -0.00005
Loss: 863.195, Residuals: -0.00005
Loss: 862.748, Residuals: -0.00004
Loss: 862.501, Residuals: -0.00004
Loss: 862.157, 

Loss: 15.087, Residuals: -0.00003
Loss: 14.778, Residuals: -0.00004
Loss: 14.362, Residuals: 0.00013
Loss: 14.046, Residuals: -0.00027
Loss: 13.880, Residuals: -0.00046
Loss: 13.629, Residuals: -0.00048
Loss: 13.253, Residuals: -0.00050
Loss: 13.123, Residuals: -0.00033
Loss: 12.907, Residuals: -0.00030
Loss: 12.605, Residuals: -0.00036
Loss: 12.531, Residuals: 0.00022
Loss: 12.396, Residuals: 0.00022
Loss: 12.171, Residuals: 0.00025
Loss: 12.028, Residuals: 0.00033
Loss: 11.806, Residuals: 0.00023
Loss: 11.563, Residuals: -0.00016
Loss: 11.545, Residuals: 0.00001
Loss: 11.399, Residuals: 0.00003
Loss: 11.185, Residuals: -0.00007
Loss: 10.922, Residuals: -0.00034
Loss: 10.908, Residuals: -0.00002
Loss: 10.883, Residuals: -0.00002
Loss: 10.840, Residuals: -0.00003
Loss: 10.770, Residuals: -0.00003
Loss: 10.759, Residuals: -0.00004
Loss: 10.737, Residuals: -0.00004
Loss: 10.695, Residuals: -0.00004
Loss: 10.625, Residuals: -0.00006
Loss: 10.523, Residuals: -0.00008
Loss: 10.379, Residual

Loss: 242.404, Residuals: 0.00005
Loss: 242.391, Residuals: 0.00005
Loss: 242.385, Residuals: 0.00005
Loss: 242.378, Residuals: 0.00005
Updating precision...
Evidence 3240.792
Loss: 592.760, Residuals: 0.00001
Loss: 587.852, Residuals: 0.00001
Loss: 583.993, Residuals: -0.00001
Loss: 580.698, Residuals: 0.00002
Loss: 579.441, Residuals: 0.00001
Loss: 577.887, Residuals: 0.00001
Loss: 576.548, Residuals: 0.00001
Loss: 575.703, Residuals: 0.00002
Loss: 574.731, Residuals: 0.00002
Loss: 574.146, Residuals: 0.00002
Loss: 573.461, Residuals: 0.00002
Loss: 573.164, Residuals: 0.00002
Loss: 572.722, Residuals: 0.00002
Loss: 572.547, Residuals: 0.00002
Loss: 572.335, Residuals: 0.00002
Loss: 572.089, Residuals: 0.00002
Loss: 571.920, Residuals: 0.00002
Loss: 571.736, Residuals: 0.00002
Loss: 571.642, Residuals: 0.00002
Loss: 571.522, Residuals: 0.00002
Loss: 571.425, Residuals: 0.00002
Loss: 571.346, Residuals: 0.00002
Loss: 571.288, Residuals: 0.00002
Loss: 571.222, Residuals: 0.00002
Loss: 5

2024-10-08 16:47:10.737746: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-08 16:47:11.059261: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-08 16:47:11.631888: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-08 16:47:11.938015: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-08 16:47:12.198602: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-08 16:47:12.473259: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-08 16:47:12.747367: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-08 16:47:13.029787: W external/xla/xla/s

# Train on DTL 0, 1 to predict held-out DTL 0, 1, 2, 3

In [6]:
species = ['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs', 'CHabs',
           'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs', 'RIabs']
controls = ['AcGum', 'ArGal', 'Inulin', 'Pectin', 'Starch', 'Xylan']
metabolites = ['pH', 'Lactate', 'Butyrate', 'Acetate']

# concatenate all observed and all system variables 
observed = np.concatenate((np.array(species), np.array(metabolites)))
system_variables = np.concatenate((np.array(species), np.array(metabolites), np.array(controls)))
system_variables

array(['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs',
       'CHabs', 'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs',
       'RIabs', 'pH', 'Lactate', 'Butyrate', 'Acetate', 'AcGum', 'ArGal',
       'Inulin', 'Pectin', 'Starch', 'Xylan'], dtype='<U8')

In [7]:
# df for training 
df = pd.concat((df_exp0, df_exp1))

In [8]:
all_treatments = df.Experiments.values
unique_treatments = np.unique(all_treatments) 

n_splits = 20
kf = KFold(n_splits=n_splits, shuffle=True, random_state=21)
k_fold_df = pd.DataFrame()
for i, (train_index, test_index) in enumerate(kf.split(unique_treatments)):

    # split into train and test
    train_index = np.in1d(all_treatments, unique_treatments[train_index])
    test_index  = np.in1d(all_treatments, unique_treatments[test_index])
    train_df = df.iloc[train_index].copy()
    test_df  = df.iloc[test_index].copy() 
    
    # add monoculture data
    train_df = pd.concat((df_mono, train_df))
    
    # scale data 
    scaler = MinMaxScaler(observed, system_variables)
    scaler.fit(train_df)
    train_df_scaled = scaler.transform(train_df.copy())
    test_df_scaled = scaler.transform(test_df.copy())
    
    # format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
    train_data = format_data(train_df, species, metabolites, controls, observed=observed)
    train_data_scaled = format_data(train_df_scaled, species, metabolites, controls, observed=observed)
    test_data = format_data(test_df, species, metabolites, controls, observed=observed)
    test_data_scaled = format_data(test_df_scaled, species, metabolites, controls, observed=observed)

    # instantiate model
    brnn = miRNN(n_species=len(species), 
                 n_metabolites=len(metabolites), 
                 n_controls=len(controls), 
                 n_hidden=32)
    
    # fit model
    brnn.fit(train_data_scaled, 
             alpha_0=0., alpha_1=1.,
             evd_tol=1e-3)

    # make predictions
    predictions = scaler.inverse_transform(brnn.predict(test_data_scaled))

    # save predictions
    pred_df = pd.DataFrame()
    for (T, X, U, Y, exp_names), (_, preds, stdvs, exp_names_pred) in zip(test_data, predictions):

        # save species predictions for each experimental condition
        for i, exp_name in enumerate(exp_names):
            # init dataframe
            pred_df_exp = pd.DataFrame()

            # insert exp name
            pred_df_exp["Experiments"] = [exp_name]*len(T[i])
            pred_df_exp["Time"] = T[i]

            for j, s in enumerate(observed):
                pred_df_exp[s + " true"] = Y[i,:,j]
                pred_df_exp[s + " pred"] = preds[i,:,j]
                pred_df_exp[s + " stdv"] = stdvs[i,:,j]

            # append to test prediction dataframe
            pred_df = pd.concat((pred_df, pred_df_exp))
    k_fold_df = pd.concat((k_fold_df, pred_df))
    k_fold_df.to_csv(f"kfold/MiRNN_20_fold_32h_dtl_1.csv", index=False)
    
# train on dtl 0,1 to predict dtl 2,3
train_df = pd.concat((df_mono, df_exp0, df_exp1))
test_df = pd.concat((df_exp2, df_exp3))

# scale data 
scaler = MinMaxScaler(observed, system_variables)
scaler.fit(train_df)
train_df_scaled = scaler.transform(train_df.copy())
test_df_scaled = scaler.transform(test_df.copy())

# format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
train_data = format_data(train_df, species, metabolites, controls, observed=observed)
train_data_scaled = format_data(train_df_scaled, species, metabolites, controls, observed=observed)
test_data = format_data(test_df, species, metabolites, controls, observed=observed)
test_data_scaled = format_data(test_df_scaled, species, metabolites, controls, observed=observed)

# instantiate model
brnn = miRNN(n_species=len(species), 
             n_metabolites=len(metabolites), 
             n_controls=len(controls), 
             n_hidden=32)

# fit model
brnn.fit(train_data_scaled, 
         alpha_0=0., alpha_1=1.,
         evd_tol=1e-3)

# make predictions
predictions = scaler.inverse_transform(brnn.predict(test_data_scaled))

# save predictions
pred_df = pd.DataFrame()
for (T, X, U, Y, exp_names), (_, preds, stdvs, exp_names_pred) in zip(test_data, predictions):

    # save species predictions for each experimental condition
    for i, exp_name in enumerate(exp_names):
        # init dataframe
        pred_df_exp = pd.DataFrame()

        # insert exp name
        pred_df_exp["Experiments"] = [exp_name]*len(T[i])
        pred_df_exp["Time"] = T[i]

        for j, s in enumerate(observed):
            pred_df_exp[s + " true"] = Y[i,:,j]
            pred_df_exp[s + " pred"] = preds[i,:,j]
            pred_df_exp[s + " stdv"] = stdvs[i,:,j]

        # append to test prediction dataframe
        pred_df = pd.concat((pred_df, pred_df_exp))
k_fold_df = pd.concat((k_fold_df, pred_df))
k_fold_df.to_csv(f"kfold/MiRNN_20_fold_32h_dtl_1.csv", index=False)

Total measurements: 9291, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 559.789, Residuals: 0.00080
Loss: 535.328, Residuals: 0.00222
Loss: 491.739, Residuals: -0.00142
Loss: 475.601, Residuals: -0.00183
Loss: 445.379, Residuals: -0.00196
Loss: 431.245, Residuals: -0.00092
Loss: 404.600, Residuals: 0.00011
Loss: 366.535, Residuals: 0.00162
Loss: 343.219, Residuals: 0.00095
Loss: 334.845, Residuals: 0.00035
Loss: 333.844, Residuals: 0.00021
Loss: 324.223, Residuals: 0.00024
Loss: 306.321, Residuals: 0.00034
Loss: 283.257, Residuals: -0.00075
Loss: 274.456, Residuals: -0.00147
Loss: 273.360, Residuals: -0.00151
Loss: 263.908, Residuals: -0.00138
Loss: 250.280, Residuals: -0.00089
Loss: 249.708, Residuals: -0.00091
Loss: 244.401, Residuals: -0.00086
Loss: 235.243, Residuals: -0.00062
Loss: 223.461, Residuals: 0.00032
Loss: 222.784, Residuals: 0.00030
Loss: 217.275, Residuals: 0.00016
Loss: 209.043, Residuals: -0.00019
Loss: 208.643, Residuals: -0.00017
Loss: 205.171, 

2024-10-08 16:56:11.868296: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 9270, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 553.721, Residuals: 0.00003
Loss: 522.684, Residuals: -0.00386
Loss: 506.141, Residuals: -0.00025
Loss: 474.716, Residuals: 0.00009
Loss: 460.921, Residuals: -0.00207
Loss: 407.946, Residuals: -0.00082
Loss: 394.618, Residuals: 0.00020
Loss: 375.022, Residuals: 0.00188
Loss: 364.583, Residuals: -0.00089
Loss: 346.369, Residuals: -0.00074
Loss: 324.915, Residuals: 0.00068
Loss: 319.366, Residuals: 0.00023
Loss: 311.784, Residuals: -0.00017
Loss: 299.441, Residuals: -0.00098
Loss: 281.559, Residuals: -0.00086
Loss: 280.098, Residuals: -0.00094
Loss: 277.314, Residuals: -0.00101
Loss: 272.064, Residuals: -0.00114
Loss: 262.599, Residuals: -0.00147
Loss: 248.525, Residuals: -0.00133
Loss: 233.830, Residuals: -0.00034
Loss: 231.088, Residuals: -0.00037
Loss: 226.929, Residuals: 0.00013
Loss: 219.765, Residuals: -0.00016
Loss: 208.559, Residuals: -0.00045
Loss: 200.249, Residuals: -0.00057
Loss: 190.

2024-10-08 17:02:53.512798: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 9186, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 546.100, Residuals: 0.00006
Loss: 514.862, Residuals: -0.00333
Loss: 506.906, Residuals: -0.00199
Loss: 447.599, Residuals: 0.00258
Loss: 442.808, Residuals: 0.00340
Loss: 433.645, Residuals: 0.00242
Loss: 391.682, Residuals: -0.00080
Loss: 366.308, Residuals: -0.00141
Loss: 335.051, Residuals: 0.00095
Loss: 318.581, Residuals: -0.00180
Loss: 314.532, Residuals: -0.00183
Loss: 308.383, Residuals: -0.00046
Loss: 297.491, Residuals: -0.00060
Loss: 278.998, Residuals: -0.00049
Loss: 262.114, Residuals: -0.00183
Loss: 257.875, Residuals: -0.00177
Loss: 251.631, Residuals: 0.00003
Loss: 242.485, Residuals: -0.00094
Loss: 228.003, Residuals: -0.00090
Loss: 211.821, Residuals: -0.00108
Loss: 205.209, Residuals: -0.00081
Loss: 194.883, Residuals: -0.00079
Loss: 183.264, Residuals: 0.00029
Loss: 181.491, Residuals: -0.00113
Loss: 178.208, Residuals: -0.00102
Loss: 172.402, Residuals: -0.00077
Loss: 163.

...

2024-10-08 19:25:31.918742: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 9208, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 548.099, Residuals: 0.00001
Loss: 517.260, Residuals: -0.00302
Loss: 500.937, Residuals: -0.00083
Loss: 469.648, Residuals: -0.00026
Loss: 439.480, Residuals: -0.00425
Loss: 398.512, Residuals: 0.00231
Loss: 389.533, Residuals: -0.00121
Loss: 373.471, Residuals: -0.00006
Loss: 347.730, Residuals: 0.00116
Loss: 340.350, Residuals: -0.00001
Loss: 328.488, Residuals: -0.00043
Loss: 319.204, Residuals: -0.00199
Loss: 315.331, Residuals: -0.00224
Loss: 314.136, Residuals: -0.00230
Loss: 302.772, Residuals: -0.00204
Loss: 283.052, Residuals: -0.00156
Loss: 267.379, Residuals: -0.00075
Loss: 261.542, Residuals: -0.00092
Loss: 258.781, Residuals: -0.00105
Loss: 253.671, Residuals: -0.00092
Loss: 244.509, Residuals: -0.00096
Loss: 229.991, Residuals: -0.00087
Loss: 218.582, Residuals: -0.00181
Loss: 217.485, Residuals: -0.00170
Loss: 215.453, Residuals: -0.00155
Loss: 211.805, Residuals: -0.00144
Loss: 

2024-10-08 19:32:53.522135: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 9709, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 576.349, Residuals: 0.00008
Loss: 543.445, Residuals: -0.00349
Loss: 526.305, Residuals: -0.00045
Loss: 493.792, Residuals: -0.00007
Loss: 479.473, Residuals: -0.00180
Loss: 424.839, Residuals: -0.00064
Loss: 411.124, Residuals: 0.00026
Loss: 390.932, Residuals: 0.00197
Loss: 363.268, Residuals: 0.00080
Loss: 355.345, Residuals: -0.00001
Loss: 341.518, Residuals: -0.00052
Loss: 336.586, Residuals: -0.00008
Loss: 327.269, Residuals: -0.00054
Loss: 310.503, Residuals: -0.00067
Loss: 289.394, Residuals: -0.00056
Loss: 288.075, Residuals: -0.00067
Loss: 277.863, Residuals: -0.00120
Loss: 264.921, Residuals: -0.00134
Loss: 264.137, Residuals: -0.00134
Loss: 257.188, Residuals: -0.00120
Loss: 245.449, Residuals: -0.00106
Loss: 234.451, Residuals: -0.00049
Loss: 222.065, Residuals: -0.00056
Loss: 221.037, Residuals: -0.00050
Loss: 219.131, Residuals: -0.00055
Loss: 215.628, Residuals: -0.00068
Loss: 2

2024-10-08 19:40:10.189517: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-08 19:40:10.581969: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-08 19:40:10.978744: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-08 19:40:11.358840: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-08 19:40:11.733542: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-08 19:40:12.101822: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-08 19:40:12.743809: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-08 19:40:13.106718: W external/xla/xla/s

# Train on DTL 0, 1, 2 to predict held-out DTL 0, 1, 2, 3

In [9]:
# df for training 
df = pd.concat((df_exp0, df_exp1, df_exp2))

In [10]:
all_treatments = df.Experiments.values
unique_treatments = np.unique(all_treatments) 

n_splits = 20
kf = KFold(n_splits=n_splits, shuffle=True, random_state=21)
k_fold_df = pd.DataFrame()
for i, (train_index, test_index) in enumerate(kf.split(unique_treatments)):

    # split into train and test
    train_index = np.in1d(all_treatments, unique_treatments[train_index])
    test_index  = np.in1d(all_treatments, unique_treatments[test_index])
    train_df = df.iloc[train_index].copy()
    test_df  = df.iloc[test_index].copy() 
    
    # add monoculture data
    train_df = pd.concat((df_mono, train_df))
    
    # scale data 
    scaler = MinMaxScaler(observed, system_variables)
    scaler.fit(train_df)
    train_df_scaled = scaler.transform(train_df.copy())
    test_df_scaled = scaler.transform(test_df.copy())
    
    # format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
    train_data = format_data(train_df, species, metabolites, controls, observed=observed)
    train_data_scaled = format_data(train_df_scaled, species, metabolites, controls, observed=observed)
    test_data = format_data(test_df, species, metabolites, controls, observed=observed)
    test_data_scaled = format_data(test_df_scaled, species, metabolites, controls, observed=observed)

    # instantiate model
    brnn = miRNN(n_species=len(species), 
                 n_metabolites=len(metabolites), 
                 n_controls=len(controls), 
                 n_hidden=32)
    # fit model
    brnn.fit(train_data_scaled, 
             alpha_0=0., alpha_1=1.,
             evd_tol=1e-3)

    # make predictions
    predictions = scaler.inverse_transform(brnn.predict(test_data_scaled))

    # save predictions
    pred_df = pd.DataFrame()
    for (T, X, U, Y, exp_names), (_, preds, stdvs, exp_names_pred) in zip(test_data, predictions):

        # save species predictions for each experimental condition
        for i, exp_name in enumerate(exp_names):
            # init dataframe
            pred_df_exp = pd.DataFrame()

            # insert exp name
            pred_df_exp["Experiments"] = [exp_name]*len(T[i])
            pred_df_exp["Time"] = T[i]

            for j, s in enumerate(observed):
                pred_df_exp[s + " true"] = Y[i,:,j]
                pred_df_exp[s + " pred"] = preds[i,:,j]
                pred_df_exp[s + " stdv"] = stdvs[i,:,j]

            # append to test prediction dataframe
            pred_df = pd.concat((pred_df, pred_df_exp))
    k_fold_df = pd.concat((k_fold_df, pred_df))
    k_fold_df.to_csv(f"kfold/MiRNN_20_fold_32h_dtl_2.csv", index=False)
    
# train on dtl 0,1,2 to predict dtl 3
train_df = pd.concat((df_mono, df_exp0, df_exp1, df_exp2))
test_df = df_exp3.copy()

# scale data 
scaler = MinMaxScaler(observed, system_variables)
scaler.fit(train_df)
train_df_scaled = scaler.transform(train_df.copy())
test_df_scaled = scaler.transform(test_df.copy())

# format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
train_data = format_data(train_df, species, metabolites, controls, observed=observed)
train_data_scaled = format_data(train_df_scaled, species, metabolites, controls, observed=observed)
test_data = format_data(test_df, species, metabolites, controls, observed=observed)
test_data_scaled = format_data(test_df_scaled, species, metabolites, controls, observed=observed)

# instantiate model
brnn = miRNN(n_species=len(species), 
             n_metabolites=len(metabolites), 
             n_controls=len(controls), 
             n_hidden=32)

# fit model
brnn.fit(train_data_scaled, 
         alpha_0=0., alpha_1=1.,
         evd_tol=1e-3)

# make predictions
predictions = scaler.inverse_transform(brnn.predict(test_data_scaled))

# save predictions
pred_df = pd.DataFrame()
for (T, X, U, Y, exp_names), (_, preds, stdvs, exp_names_pred) in zip(test_data, predictions):

    # save species predictions for each experimental condition
    for i, exp_name in enumerate(exp_names):
        # init dataframe
        pred_df_exp = pd.DataFrame()

        # insert exp name
        pred_df_exp["Experiments"] = [exp_name]*len(T[i])
        pred_df_exp["Time"] = T[i]

        for j, s in enumerate(observed):
            pred_df_exp[s + " true"] = Y[i,:,j]
            pred_df_exp[s + " pred"] = preds[i,:,j]
            pred_df_exp[s + " stdv"] = stdvs[i,:,j]

        # append to test prediction dataframe
        pred_df = pd.concat((pred_df, pred_df_exp))
k_fold_df = pd.concat((k_fold_df, pred_df))
k_fold_df.to_csv(f"kfold/MiRNN_20_fold_32h_dtl_2.csv", index=False)

Total measurements: 16234, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 960.304, Residuals: -0.00359
Loss: 916.373, Residuals: -0.00067
Loss: 904.557, Residuals: 0.00095
Loss: 810.704, Residuals: -0.00117
Loss: 729.387, Residuals: -0.00061
Loss: 696.408, Residuals: -0.00018
Loss: 644.365, Residuals: 0.00339
Loss: 624.341, Residuals: 0.00083
Loss: 588.048, Residuals: 0.00107
Loss: 565.916, Residuals: -0.00094
Loss: 559.132, Residuals: 0.00184
Loss: 506.120, Residuals: 0.00009
Loss: 504.412, Residuals: -0.00004
Loss: 488.463, Residuals: -0.00012
Loss: 459.609, Residuals: -0.00044
Loss: 428.007, Residuals: -0.00196
Loss: 417.198, Residuals: -0.00130
Loss: 398.534, Residuals: -0.00111
Loss: 374.777, Residuals: 0.00000
Loss: 372.152, Residuals: 0.00002
Loss: 367.791, Residuals: -0.00022
Loss: 361.154, Residuals: -0.00028
Loss: 350.100, Residuals: -0.00037
Loss: 343.817, Residuals: -0.00054
Loss: 333.702, Residuals: -0.00065
Loss: 333.436, Residuals: -0.00064
Loss: 324.

2024-10-08 19:53:14.571557: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 16173, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 973.072, Residuals: -0.00265
Loss: 929.582, Residuals: -0.00018
Loss: 854.174, Residuals: -0.00161
Loss: 846.438, Residuals: -0.00025
Loss: 763.628, Residuals: 0.00084
Loss: 715.864, Residuals: -0.00024
Loss: 671.427, Residuals: 0.00143
Loss: 614.920, Residuals: 0.00168
Loss: 608.914, Residuals: 0.00127
Loss: 597.902, Residuals: 0.00065
Loss: 578.679, Residuals: 0.00085
Loss: 548.108, Residuals: 0.00044
Loss: 500.221, Residuals: -0.00103
Loss: 494.815, Residuals: 0.00021
Loss: 455.493, Residuals: -0.00015
Loss: 453.563, Residuals: -0.00025
Loss: 437.235, Residuals: -0.00044
Loss: 412.107, Residuals: -0.00138
Loss: 411.407, Residuals: -0.00137
Loss: 404.844, Residuals: -0.00132
Loss: 392.808, Residuals: -0.00116
Loss: 373.493, Residuals: -0.00108
Loss: 362.879, Residuals: -0.00072
Loss: 361.462, Residuals: -0.00077
Loss: 359.128, Residuals: -0.00086
Loss: 354.888, Residuals: -0.00083
Loss: 347.

2024-10-08 20:02:03.782132: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 16309, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 974.867, Residuals: -0.00314
Loss: 930.361, Residuals: -0.00047
Loss: 853.186, Residuals: -0.00158
Loss: 788.430, Residuals: -0.00472
Loss: 710.995, Residuals: 0.00011
Loss: 685.083, Residuals: -0.00053
Loss: 640.206, Residuals: 0.00030
Loss: 624.503, Residuals: 0.00088
Loss: 599.509, Residuals: 0.00069
Loss: 560.693, Residuals: -0.00035
Loss: 558.285, Residuals: -0.00047
Loss: 536.738, Residuals: -0.00043
Loss: 502.208, Residuals: -0.00085
Loss: 483.035, Residuals: -0.00017
Loss: 476.171, Residuals: -0.00060
Loss: 467.625, Residuals: -0.00060
Loss: 452.873, Residuals: -0.00091
Loss: 428.962, Residuals: -0.00166
Loss: 421.873, Residuals: -0.00193
Loss: 409.117, Residuals: -0.00178
Loss: 388.111, Residuals: -0.00128
Loss: 370.547, Residuals: -0.00029
Loss: 365.765, Residuals: -0.00081
Loss: 358.061, Residuals: -0.00118
Loss: 348.069, Residuals: -0.00108
Loss: 347.764, Residuals: -0.00106
Loss: 

...

Loss: 7782.339, Residuals: -0.00031
Loss: 7775.104, Residuals: -0.00034
Loss: 7771.791, Residuals: -0.00032
Loss: 7769.208, Residuals: -0.00033
Loss: 7764.978, Residuals: -0.00034
Loss: 7763.114, Residuals: -0.00034
Loss: 7760.617, Residuals: -0.00034
Loss: 7757.463, Residuals: -0.00034
Loss: 7755.530, Residuals: -0.00034
Loss: 7752.675, Residuals: -0.00034
Loss: 7752.109, Residuals: -0.00034
Loss: 7752.037, Residuals: -0.00033
Loss: 7751.906, Residuals: -0.00033
Loss: 7751.814, Residuals: -0.00033
Updating precision...
Evidence 24384.416
Loss: 7923.479, Residuals: -0.00035
Loss: 7919.518, Residuals: -0.00036
Loss: 7918.532, Residuals: -0.00036
Loss: 7916.748, Residuals: -0.00036
Loss: 7916.072, Residuals: -0.00036
Updating precision...
Evidence 24468.904
Updating precision...
Evidence 24501.885
Updating precision...
Evidence 24515.850
Pass count  1


2024-10-09 00:00:15.887540: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 17069, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1015.335, Residuals: -0.00197
Loss: 968.948, Residuals: 0.00057
Loss: 956.187, Residuals: 0.00212
Loss: 856.164, Residuals: -0.00069
Loss: 772.167, Residuals: -0.00071
Loss: 738.122, Residuals: 0.00006
Loss: 684.784, Residuals: 0.00325
Loss: 664.092, Residuals: 0.00113
Loss: 629.068, Residuals: -0.00069
Loss: 615.369, Residuals: 0.00187
Loss: 588.314, Residuals: 0.00096
Loss: 547.282, Residuals: -0.00028
Loss: 540.471, Residuals: 0.00015
Loss: 527.415, Residuals: -0.00016
Loss: 502.971, Residuals: -0.00041
Loss: 462.368, Residuals: -0.00163
Loss: 461.115, Residuals: -0.00165
Loss: 449.729, Residuals: -0.00166
Loss: 430.198, Residuals: -0.00146
Loss: 405.426, Residuals: -0.00056
Loss: 403.065, Residuals: 0.00011
Loss: 398.438, Residuals: 0.00001
Loss: 390.109, Residuals: -0.00016
Loss: 376.097, Residuals: -0.00040
Loss: 364.539, Residuals: -0.00062
Loss: 362.282, Residuals: 0.00017
Loss: 358.22

Loss: 6173.838, Residuals: -0.00048
Loss: 6173.772, Residuals: -0.00048
Loss: 6173.665, Residuals: -0.00048
Loss: 6173.505, Residuals: -0.00048
Loss: 6173.467, Residuals: -0.00048
Updating precision...
Evidence 25021.273
Loss: 7822.515, Residuals: -0.00050
Loss: 7810.245, Residuals: -0.00048
Loss: 7808.783, Residuals: -0.00049
Loss: 7805.581, Residuals: -0.00049
Loss: 7802.072, Residuals: -0.00049
Loss: 7797.235, Residuals: -0.00050
Loss: 7788.082, Residuals: -0.00049
Loss: 7787.789, Residuals: -0.00049
Loss: 7783.886, Residuals: -0.00049
Loss: 7780.340, Residuals: -0.00048
Loss: 7775.824, Residuals: -0.00047
Loss: 7770.516, Residuals: -0.00048
Updating precision...
Evidence 25804.855
Loss: 8231.343, Residuals: -0.00048
Loss: 8224.505, Residuals: -0.00049
Loss: 8216.232, Residuals: -0.00052
Loss: 8210.529, Residuals: -0.00050
Loss: 8207.726, Residuals: -0.00050
Loss: 8205.181, Residuals: -0.00049
Loss: 8203.992, Residuals: -0.00049
Loss: 8201.407, Residuals: -0.00049
Loss: 8197.113, Re

2024-10-09 00:15:31.367057: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-09 00:15:31.728624: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-09 00:15:32.326548: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-09 00:15:32.653878: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
2024-10-09 00:15:32.967337: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


In [11]:
# df for training 
df = pd.concat((df_exp0, df_exp1, df_exp2, df_exp3))

In [12]:
all_treatments = df.Experiments.values
unique_treatments = np.unique(all_treatments) 

n_splits = 20
kf = KFold(n_splits=n_splits, shuffle=True, random_state=21)
k_fold_df = pd.DataFrame()
for i, (train_index, test_index) in enumerate(kf.split(unique_treatments)):

    # split into train and test
    train_index = np.in1d(all_treatments, unique_treatments[train_index])
    test_index  = np.in1d(all_treatments, unique_treatments[test_index])
    train_df = df.iloc[train_index].copy()
    test_df  = df.iloc[test_index].copy() 
    
    # add monoculture data
    train_df = pd.concat((df_mono, train_df))
    
    # scale data 
    scaler = MinMaxScaler(observed, system_variables)
    scaler.fit(train_df)
    train_df_scaled = scaler.transform(train_df.copy())
    test_df_scaled = scaler.transform(test_df.copy())
    
    # format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
    train_data = format_data(train_df, species, metabolites, controls, observed=observed)
    train_data_scaled = format_data(train_df_scaled, species, metabolites, controls, observed=observed)
    test_data = format_data(test_df, species, metabolites, controls, observed=observed)
    test_data_scaled = format_data(test_df_scaled, species, metabolites, controls, observed=observed)

    # instantiate model
    brnn = miRNN(n_species=len(species), 
                 n_metabolites=len(metabolites), 
                 n_controls=len(controls), 
                 n_hidden=32)
    # fit model
    brnn.fit(train_data_scaled, 
             alpha_0=0., alpha_1=1.,
             evd_tol=1e-3)

    # make predictions
    predictions = scaler.inverse_transform(brnn.predict(test_data_scaled))

    # save predictions
    pred_df = pd.DataFrame()
    for (T, X, U, Y, exp_names), (_, preds, stdvs, exp_names_pred) in zip(test_data, predictions):

        # save species predictions for each experimental condition
        for i, exp_name in enumerate(exp_names):
            # init dataframe
            pred_df_exp = pd.DataFrame()

            # insert exp name
            pred_df_exp["Experiments"] = [exp_name]*len(T[i])
            pred_df_exp["Time"] = T[i]

            for j, s in enumerate(observed):
                pred_df_exp[s + " true"] = Y[i,:,j]
                pred_df_exp[s + " pred"] = preds[i,:,j]
                pred_df_exp[s + " stdv"] = stdvs[i,:,j]

            # append to test prediction dataframe
            pred_df = pd.concat((pred_df, pred_df_exp))
    k_fold_df = pd.concat((k_fold_df, pred_df))
    k_fold_df.to_csv(f"kfold/MiRNN_20_fold_32h_dtl_3.csv", index=False)

Total measurements: 25536, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1260.841, Residuals: -0.00670
Loss: 1179.536, Residuals: -0.00278
Loss: 1152.097, Residuals: 0.00003
Loss: 1126.656, Residuals: 0.00025
Loss: 1023.105, Residuals: 0.00329
Loss: 971.608, Residuals: -0.00174
Loss: 897.712, Residuals: -0.00137
Loss: 842.996, Residuals: -0.00016
Loss: 773.649, Residuals: 0.00085
Loss: 761.236, Residuals: 0.00071
Loss: 741.280, Residuals: 0.00056
Loss: 703.710, Residuals: 0.00020
Loss: 647.690, Residuals: -0.00115
Loss: 640.898, Residuals: -0.00002
Loss: 627.711, Residuals: -0.00013
Loss: 603.344, Residuals: -0.00051
Loss: 563.304, Residuals: -0.00061
Loss: 553.913, Residuals: -0.00107
Loss: 536.702, Residuals: -0.00103
Loss: 508.445, Residuals: -0.00025
Loss: 507.975, Residuals: -0.00046
Loss: 503.500, Residuals: -0.00045
Loss: 496.025, Residuals: -0.00085
Loss: 483.988, Residuals: -0.00063
Loss: 470.218, Residuals: -0.00112
Loss: 468.570, Residuals: -0.00077
Loss

2024-10-09 00:28:37.692490: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 25457, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1260.373, Residuals: -0.00637
Loss: 1181.546, Residuals: -0.00276
Loss: 1154.392, Residuals: -0.00015
Loss: 1126.962, Residuals: 0.00112
Loss: 1019.915, Residuals: 0.00403
Loss: 994.032, Residuals: -0.00262
Loss: 948.317, Residuals: -0.00091
Loss: 865.172, Residuals: 0.00080
Loss: 834.858, Residuals: -0.00115
Loss: 781.411, Residuals: -0.00057
Loss: 750.237, Residuals: -0.00116
Loss: 704.149, Residuals: -0.00034
Loss: 649.818, Residuals: -0.00383
Loss: 638.762, Residuals: -0.00074
Loss: 617.366, Residuals: -0.00088
Loss: 581.036, Residuals: -0.00114
Loss: 542.145, Residuals: -0.00090
Loss: 540.797, Residuals: -0.00092
Loss: 540.308, Residuals: -0.00092
Loss: 522.949, Residuals: -0.00062
Loss: 498.618, Residuals: -0.00091
Loss: 496.824, Residuals: -0.00118
Loss: 493.539, Residuals: -0.00111
Loss: 487.773, Residuals: -0.00107
Loss: 479.774, Residuals: -0.00100
Loss: 468.025, Residuals: -0.00101


2024-10-09 00:43:27.311630: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 25581, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1265.225, Residuals: -0.00600
Loss: 1184.782, Residuals: -0.00290
Loss: 1154.829, Residuals: -0.00016
Loss: 1125.531, Residuals: 0.00173
Loss: 1017.586, Residuals: 0.00494
Loss: 991.724, Residuals: -0.00291
Loss: 946.719, Residuals: -0.00018
Loss: 863.830, Residuals: 0.00014
Loss: 793.543, Residuals: 0.00201
Loss: 753.091, Residuals: 0.00076
Loss: 713.321, Residuals: -0.00117
Loss: 708.430, Residuals: -0.00125
Loss: 701.979, Residuals: -0.00011
Loss: 648.261, Residuals: -0.00092
Loss: 621.031, Residuals: -0.00309
Loss: 583.812, Residuals: -0.00126
Loss: 578.254, Residuals: -0.00142
Loss: 540.578, Residuals: -0.00065
Loss: 536.638, Residuals: -0.00068
Loss: 530.902, Residuals: -0.00119
Loss: 521.563, Residuals: -0.00127
Loss: 507.219, Residuals: -0.00153
Loss: 487.783, Residuals: -0.00063
Loss: 486.479, Residuals: -0.00063
Loss: 486.258, Residuals: -0.00071
Loss: 484.187, Residuals: -0.00062
Lo

...

2024-10-09 04:09:19.715880: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 25678, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1272.518, Residuals: -0.00631
Loss: 1190.975, Residuals: -0.00276
Loss: 1162.879, Residuals: -0.00009
Loss: 1134.805, Residuals: 0.00147
Loss: 1028.439, Residuals: 0.00433
Loss: 1002.479, Residuals: -0.00262
Loss: 956.809, Residuals: -0.00058
Loss: 872.883, Residuals: -0.00059
Loss: 839.846, Residuals: -0.00066
Loss: 780.709, Residuals: -0.00076
Loss: 727.923, Residuals: -0.00465
Loss: 712.276, Residuals: -0.00463
Loss: 691.763, Residuals: -0.00167
Loss: 669.956, Residuals: -0.00122
Loss: 635.049, Residuals: -0.00146
Loss: 590.714, Residuals: -0.00242
Loss: 588.092, Residuals: -0.00234
Loss: 584.072, Residuals: -0.00155
Loss: 577.377, Residuals: -0.00174
Loss: 565.259, Residuals: -0.00173
Loss: 542.607, Residuals: -0.00143
Loss: 542.294, Residuals: -0.00141
Loss: 531.079, Residuals: -0.00153
Loss: 513.344, Residuals: -0.00162
Loss: 493.317, Residuals: -0.00129
Loss: 491.749, Residuals: -0.0012

2024-10-09 04:21:42.820408: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 25476, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1253.861, Residuals: -0.00616
Loss: 1174.304, Residuals: -0.00246
Loss: 1147.297, Residuals: 0.00014
Loss: 1119.849, Residuals: 0.00116
Loss: 1015.119, Residuals: 0.00402
Loss: 990.056, Residuals: -0.00257
Loss: 945.743, Residuals: -0.00073
Loss: 862.543, Residuals: -0.00066
Loss: 828.302, Residuals: -0.00085
Loss: 768.915, Residuals: -0.00052
Loss: 738.274, Residuals: -0.00109
Loss: 685.586, Residuals: -0.00188
Loss: 673.168, Residuals: -0.00065
Loss: 617.920, Residuals: -0.00164
Loss: 613.034, Residuals: -0.00000
Loss: 575.564, Residuals: -0.00129
Loss: 566.825, Residuals: -0.00063
Loss: 550.845, Residuals: -0.00052
Loss: 523.936, Residuals: -0.00045
Loss: 523.366, Residuals: -0.00046
Loss: 518.512, Residuals: -0.00060
Loss: 511.003, Residuals: -0.00083
Loss: 499.724, Residuals: -0.00083
Loss: 483.757, Residuals: -0.00071
Loss: 478.527, Residuals: -0.00074
Loss: 478.416, Residuals: -0.00074


2024-10-09 04:34:44.308878: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
